# Семинар 1 — дополнительно

Здесь то, что не влезло в 80 минут семинара. Читать необязательно и
можно вразнобой: разделы независимы, каждый отвечает на один вопрос.

| # | Вопрос |
|---|---|
| 1 | Почему у колеса такое странное имя |
| 2 | Почему `pip install pillow`, а `import PIL` |
| 3 | Зачем класть код в `src/` |
| 4 | Как устроены номера версий |
| 5 | Как читать строку зависимости |
| 6 | Что происходит при `pip install -e .` |
| 7 | Что ещё лежит в `dist-info` |
| 8 | Зачем нужно столько build-backend'ов |
| 9 | Как публиковать из CI без токенов |
| 10 | Почему `pip install` — это опасно |

Базовое — в [`seminar.ipynb`](seminar.ipynb).

In [ ]:
import pathlib
import shutil
import subprocess
import sys
import tempfile
import zipfile

WORK = pathlib.Path(tempfile.mkdtemp(prefix="seminar01-"))
UV = shutil.which("uv")

print("песочница:", WORK)


def sh(command: str, cwd: pathlib.Path = WORK, quiet: bool = False) -> None:
    """Выполнить shell-команду в песочнице и показать её вывод."""
    result = subprocess.run(
        command, shell=True, cwd=cwd, capture_output=True, text=True
    )
    if quiet:
        return
    if result.stdout.strip():
        print(result.stdout, end="")
    if result.stderr.strip():
        print("--- stderr ---")
        print(result.stderr, end="")

## 1. Почему у колеса такое странное имя

```
demo_greet_seminar-0.1.0-py3-none-any.whl
└──── имя ───────┘ └ver┘ └py3┘└none┘└any┘
                          │     │     └─ платформа
                          │     └─ ABI
                          └─ версия Python
```

Три последних куска — **теги совместимости**. По ним `pip` решает,
подходит ли колесо твоей машине.

| Тег | Значение |
|---|---|
| `py3-none-any` | чистый Python, встанет где угодно |
| `cp314-cp314-macosx_11_0_arm64` | только CPython 3.14, только Apple Silicon |
| `cp314-cp314-manylinux_2_17_x86_64` | только CPython 3.14, Linux с glibc ≥ 2.17 |

Поэтому у `numpy` под каждый релиз десятки колёс, а у нашего пакета одно.
Если подходящего колеса нет, `pip` берёт sdist и собирает на месте — вот
тут и нужен компилятор.

Заодно: имя файла и имя дистрибутива пишутся по-разному.
`demo-greet-seminar` в конфиге превращается в `demo_greet_seminar` в имени
файла — дефисы заменяются подчёркиваниями. Это нормализация, описанная в
стандарте; поэтому `my-pkg` и `my_pkg` считаются **одним** именем, и занять
оба не получится.

## 2. Почему `pip install pillow`, а `import PIL`

Две разные сущности, которые часто путают:

- **имя дистрибутива** — то, что в `name` и в `pip install`;
- **имя импорта** — папка с `__init__.py`, то, что пишешь в `import`.

Связь между ними задаёт только конфиг сборки. Совпадение — традиция.

| `pip install` | `import` |
|---|---|
| `pillow` | `PIL` |
| `scikit-learn` | `sklearn` |
| `pyyaml` | `yaml` |
| `beautifulsoup4` | `bs4` |
| `python-dateutil` | `dateutil` |
| `opencv-python` | `cv2` |
| `attrs` | `attr` |
| `pyjwt` | `jwt` |

Практическое следствие: **по `import` нельзя угадать, что ставить**.
Особенно больно при восстановлении окружения по чужому коду.

Посмотреть, что разъехалось в твоём окружении:

In [ ]:
from importlib.metadata import packages_distributions

mapping = packages_distributions()

for import_name, dists in sorted(mapping.items()):
    if import_name.startswith("_") or not import_name[0].isalpha():
        continue
    for dist in dists:
        if dist.lower().replace("-", "_") != import_name.lower():
            print(f"import {import_name:16} →  pip install {dist}")
            break

## 3. Зачем класть код в `src/`

```
flat-layout                      src-layout
.                                .
├── my_package/                  ├── src/
│   └── __init__.py              │   └── my_package/
├── tests/                       │       └── __init__.py
└── pyproject.toml               ├── tests/
                                 └── pyproject.toml
```

Разница не косметическая.

При **flat-layout** папка пакета лежит в корне проекта, а корень проекта —
это твой рабочий каталог. Значит `import my_package` сработает, **даже
если пакет не установлен**: Python найдёт папку рядом.

Звучит удобно — и ломает проверку. Тесты зеленеют по исходникам из папки,
а не по тому, что попало в колесо. Забыл включить файл в сборку — узнаешь
от пользователя.

При **src-layout** корень проекта пустой: без установки импорт падает.
Значит тесты гоняются только по установленному пакету — ровно по тому,
что получит пользователь.

Задача семинара сделана на src-layout именно поэтому: тест
`test_stopwords_shipped_with_package` проверяет реальное содержимое
установленного пакета.

## 4. Как устроены номера версий

**SemVer** — соглашение о смысле: `MAJOR.MINOR.PATCH`. Сломал
совместимость — MAJOR, добавил фичу — MINOR, починил баг — PATCH.

**PEP 440** — синтаксис, который понимает сама экосистема Python. Он шире
SemVer: есть пре-релизы, пост-релизы, dev-версии.

Главное: версии **упорядочены**, и порядок не лексикографический.

In [ ]:
from packaging.version import Version

raw = ["1.0", "1.0.1", "1.0a1", "1.0b2", "1.0rc1", "1.0.post1", "1.0.dev1", "2.0", "10.0"]
for version in sorted(raw, key=Version):
    print(" ", version)

`1.0.dev1 < 1.0a1 < 1.0b2 < 1.0rc1 < 1.0 < 1.0.post1`, и `10.0` идёт
после `2.0` — а не между `1.0` и `2.0`, как было бы при сравнении строк.

Отсюда работают спецификаторы вроде `>=1.0`: сравниваются разобранные
версии, а не строки. И `1.0rc1` **не** попадёт под `>=1.0` — пре-релизы
по умолчанию исключены.

### Где хранить версию

```toml
version = "0.1.0"                            # прямо в конфиге
```

```toml
dynamic = ["version"]                        # объявили: версия извне
[tool.hatch.version]
path = "src/demo_greet/__init__.py"          # ...вот отсюда
```

Второй вариант лучше: `__version__` доступен в рантайме и не дублируется.
Но `dynamic` — только **объявление**. Не скажешь backend'у, откуда читать,
— сборка упадёт с «Field version declared as dynamic but no source».

## 5. Как читать строку зависимости

```
requests >= 2.32, < 3.0 ; python_version < "3.12"
└─ имя ─┘ └─ спецификатор ─┘ └──── маркер окружения ────┘
```

Разберём настоящие строки тем же парсером, что использует pip:

In [ ]:
from packaging.requirements import Requirement

samples = [
    "requests>=2.32",
    "numpy>=1.26,<2.0",
    'tomli; python_version < "3.11"',
    "httpx[http2]>=0.27",
    "mypkg @ git+https://github.com/org/repo.git@a1b2c3d",
]
for line in samples:
    req = Requirement(line)
    print(f"{req.name:10} спец={str(req.specifier) or '—':16} "
          f"extras={req.extras or '—'} маркер={req.marker or '—'}")

**Операторы:**

| Запись | Смысл |
|---|---|
| `>=2.0,<3.0` | явный диапазон, самое честное |
| `~=2.4` | «совместимо»: `>=2.4, ==2.*` |
| `==2.4.*` | любой патч в пределах 2.4 |
| `!=2.4.1` | исключить сломанный релиз |

**Extras** — опциональные наборы:

```toml
[project.optional-dependencies]
http2 = ["h2>=4"]
```
```bash
pip install "mypkg[http2]"
```

**dependency-groups** — про другое: это зависимости **разработки**, они не
попадают в метаданные пакета и их нельзя поставить через `pip install
pkg[...]`. Ровно то, что нужно для `pytest` и `ruff`:

```toml
[dependency-groups]
dev = ["pytest>=8", "ruff>=0.6"]
```

### Границы и lock

`dependencies` в конфиге — **границы допустимого**, а не точные версии.
Точные фиксирует lock-файл (`uv.lock`): там весь граф с хешами.

Правило: **приложение** коммитит lock (нужна воспроизводимость),
**библиотека** — нет (нельзя навязывать точные версии тем, кто её ставит).

## 6. Что происходит при `pip install -e .`

Обычная установка копирует файлы в `site-packages` — значит правка
исходника ничего не меняет, пока не переустановишь.

Editable-установка кладёт туда не код, а указатель на твою рабочую папку.
Посмотрим, как это выглядит физически.

In [ ]:
DEMO = WORK / "demo-greet"
SRC = DEMO / "src" / "demo_greet"
(SRC / "data").mkdir(parents=True)

(SRC / "__init__.py").write_text('__version__ = "0.1.0"\n', encoding="utf-8")

# Фразы — обычный .txt внутри пакета. Запомни это место.
(SRC / "data" / "phrases.txt").write_text(
    "ru\tПривет\nen\tHello\nfr\tBonjour\n", encoding="utf-8"
)

(SRC / "core.py").write_text(
    """
from importlib.resources import files


def phrases() -> dict[str, str]:
    raw = files("demo_greet").joinpath("data", "phrases.txt").read_text(encoding="utf-8")
    return dict(line.split("\t") for line in raw.splitlines() if line)


def greet(name: str, lang: str = "ru") -> str:
    return f"{phrases()[lang]}, {name}!"
""".lstrip(),
    encoding="utf-8",
)

(SRC / "cli.py").write_text(
    """
import sys

from demo_greet.core import greet


def main() -> int:
    print(greet(sys.argv[1] if len(sys.argv) > 1 else "мир"))
    return 0
""".lstrip(),
    encoding="utf-8",
)

(DEMO / "README.md").write_text("# demo-greet\n\nДемо семинара 1.\n", encoding="utf-8")

sh(f"find {DEMO} -type f | sed 's|{DEMO}/||' | sort")

In [ ]:
PYPROJECT = """
[build-system]
requires = ["hatchling>=1.27"]
build-backend = "hatchling.build"

[project]
name = "demo-greet-seminar"
version = "0.1.0"
description = "Демонстрационный пакет семинара 1"
readme = "README.md"
requires-python = ">=3.10"
dependencies = []

[project.scripts]
demo-greet = "demo_greet.cli:main"

[tool.hatch.build.targets.wheel]
packages = ["src/demo_greet"]
"""

(DEMO / "pyproject.toml").write_text(PYPROJECT.lstrip(), encoding="utf-8")
print("pyproject.toml записан — теперь это пакет")

In [ ]:
sh(f"{UV} venv {WORK}/venv-dev", cwd=WORK, quiet=True)
sh(f"{UV} pip install --python {WORK}/venv-dev/bin/python -e {DEMO}", cwd=WORK, quiet=True)

site_packages = next((WORK / "venv-dev" / "lib").glob("python*/site-packages"))

for pth in sorted(site_packages.glob("*.pth")):
    print(f"--- {pth.name} ---")
    print(pth.read_text(encoding="utf-8").strip()[:300])
    print()

`.pth`-файл — древний механизм: при старте интерпретатора Python читает
все `.pth` из `site-packages` и добавляет указанные пути в `sys.path`
(а строки, начинающиеся с `import`, — исполняет).

Здесь внутри просто путь к `src/`. Другие backend'ы кладут туда
`import __editable___...` — маленький модуль-finder, который перехватывает
импорт конкретного пакета и не тащит в `sys.path` всю папку. Второй
вариант аккуратнее: не сделает импортируемым случайного соседа по `src/`.

Проверим, что правка видна без переустановки:

In [ ]:
core_file = DEMO / "src" / "demo_greet" / "core.py"
original = core_file.read_text(encoding="utf-8")

core_file.write_text(
    original.replace(
        'f"{phrases()[lang]}, {name}!"',
        'f"{phrases()[lang]}, {name}!!! (правка на лету)"',
    ),
    encoding="utf-8",
)

sh(f'{WORK}/venv-dev/bin/python -c "from demo_greet.core import greet; print(greet(\'мир\'))"')

core_file.write_text(original, encoding="utf-8")

Переустановки не было — вывод изменился.

Важная оговорка: `-e` **не** отслеживает изменения в `pyproject.toml`.
Добавил зависимость, поменял точку входа — нужно `pip install -e .` заново.

## 7. Что ещё лежит в `dist-info`

Кроме `METADATA` и `entry_points.txt` там есть ещё два файла.

In [ ]:
sh(f"{UV} build --wheel", cwd=DEMO, quiet=True)
wheel_path = next((DEMO / "dist").glob("*.whl"))

with zipfile.ZipFile(wheel_path) as archive:
    record_name = next(n for n in archive.namelist() if n.endswith("/RECORD"))
    for line in archive.read(record_name).decode().splitlines():
        name, digest, size = line.rsplit(",", 2)
        digest = f"{digest[:20]}…" if digest else "— (сам RECORD)"
        print(f"  {name:48} {digest:24} {size:>6}")

`RECORD` — список всех файлов с хешами и размерами. По нему
`pip uninstall` понимает, что удалять, и по нему же можно проверить,
что установленное не подменили.

In [ ]:
with zipfile.ZipFile(wheel_path) as archive:
    wheel_name = next(n for n in archive.namelist() if n.endswith("/WHEEL"))
    print(archive.read(wheel_name).decode())

`WHEEL` — про формат самого архива. `Root-Is-Purelib: true` значит, что
код чистый Python и кладётся в `purelib`. `Tag` — тот же тег
совместимости, что и в имени файла (раздел 1).

## 8. Зачем нужно столько build-backend'ов

Не мода — разные задачи.

| Backend | Когда берут |
|---|---|
| `setuptools` | легаси, C-расширения, максимум совместимости |
| `hatchling` | современный дефолт для чистого Python |
| `flit-core` | минимализм: один модуль, ноль конфига |
| `pdm-backend` | если уже живёшь в pdm |
| `maturin` | Rust через PyO3 |
| `scikit-build-core` | C/C++ через CMake |
| `meson-python` | numpy, scipy — сложные сборки |

Протокол между frontend'ом (`pip`, `uv`, `build`) и backend'ом описан
стандартом: frontend создаёт изолированное окружение, ставит туда
`requires`, импортирует `build-backend` и зовёт у него `build_wheel()`.
Frontend про твой проект не знает ничего — ни про layout, ни про
package data. Всё это знает backend, отсюда и разное поведение.

Frontend'ов тоже несколько, и они взаимозаменяемы:

```bash
uv build                        # быстрый, встроенный
uv run --with build -m build    # референсный от PyPA
pipx run build                  # то же без uv
```

## 9. Как публиковать из CI без токенов

Долгоживущий токен в секретах CI — слабое место: утёк один раз, и от
твоего имени публикуют что угодно.

**Trusted publishing** убирает токен вовсе. На PyPI регистрируется
доверенный издатель: «репозиторий `org/repo`, workflow `release.yml`,
окружение `pypi`». GitHub Actions при запуске получает короткоживущий
OIDC-токен, PyPI проверяет его и выдаёт временный доступ на несколько
минут.

```yaml
jobs:
  publish:
    runs-on: ubuntu-latest
    environment: pypi
    permissions:
      id-token: write          # ← это и заменяет секрет
    steps:
      - uses: actions/checkout@v4
      - uses: astral-sh/setup-uv@v5
      - run: uv build
      - run: uv publish --trusted-publishing always
```

Ни одного секрета в репозитории. Так публикуется большинство заметных
проектов с 2023 года.

Настроить для TestPyPI: <https://test.pypi.org/manage/account/publishing/>

## 10. Почему `pip install` — это опасно

`pip install` **выполняет код автора пакета** на твоей машине. Не
«скачивает данные», а именно выполняет.

Отсюда **typosquatting**: `requests` — настоящий, а `request` (без -s),
`requeests`, `python-requests` — вредоносные копии с похожими именами.
Реальные инциденты: `colourama` вместо `colorama` (2017), десятки копий
популярных пакетов в 2020-2024.

Как защищаться:

- сверять имя перед установкой — звучит глупо, но ловит большинство;
- `pip install --require-hashes` с зафиксированными SHA256 — pip сверит
  хеш колеса. Тот же принцип у `uv.lock`;
- не ставить пакеты из непроверенных sdist: `pip install .` из sdist
  запускает чужой `setup.py`. Это одна из причин ухода от исполняемого
  `setup.py` к декларативному `pyproject.toml`.

### Ещё про хранение токена

`~/.pypirc` — классический конфиг:

```ini
[distutils]
index-servers = testpypi

[testpypi]
repository = https://test.pypi.org/legacy/
username = __token__
password = pypi-AgENdGVzdC5weXBpLm9yZw...
```

Пароль в открытом виде. `chmod 600`, и никогда в git. Переменные
окружения безопаснее, trusted publishing (раздел 9) — ещё безопаснее.

## Что почитать

- [Python Packaging User Guide](https://packaging.python.org/) — официальный и актуальный
- [PEP 517](https://peps.python.org/pep-0517/) / [PEP 518](https://peps.python.org/pep-0518/) — протокол сборки
- [PEP 621](https://peps.python.org/pep-0621/) — метаданные в `pyproject.toml`
- [PEP 440](https://peps.python.org/pep-0440/) — версии
- [PEP 508](https://peps.python.org/pep-0508/) — синтаксис зависимостей
- [PEP 660](https://peps.python.org/pep-0660/) — editable installs
- [документация uv](https://docs.astral.sh/uv/)
- [pypi.org/classifiers](https://pypi.org/classifiers/) — допустимые классификаторы

In [ ]:
shutil.rmtree(WORK, ignore_errors=True)
print("песочница удалена:", WORK)